# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata.to_json()
print(f"Dataset title: {metadata.get('name','N/A')}")
print(f"Description: {metadata.get('description','N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs and metadata.

In [ ]:
# List record sets with their @id and name
print('Available record sets:')
for rs in dataset.record_sets:
    print(f"- @id: {rs.id}, name: {getattr(rs, 'name', 'N/A')}")

# As an example, preview fields for each record set
print('\nRecord set fields:')
for rs in dataset.record_sets:
    print(f"\nRecord Set @id: {rs.id}, name: {getattr(rs, 'name', 'N/A')}")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, name: {getattr(field, 'name', 'N/A')} (type: {getattr(field, 'data_type', 'N/A')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set ids
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Load data for each record set
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()

# Inspect one of the record sets (choose the first as example)
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    print(f"Record set @id: {record_set_id}")
    print("Columns:", df.columns.tolist())
    display(df.head(5))
else:
    print('No record sets were found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing fields, and grouping, referencing fields by their `@id`.

In [ ]:
# This block assumes at least one record set with numeric fields
import numpy as np
from IPython.display import display

if record_sets:
    rs = dataset.record_sets[0]
    df = dataframes[rs.id]
    # Find numeric fields in this record set
    numeric_fields = [field.id for field in rs.fields if field.data_type in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number']]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Example numeric field: {numeric_field}")
        if numeric_field in df.columns and not df.empty:
            # Attempt to convert to float
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
            threshold = np.nanpercentile(df[numeric_field], 75) # use 75th percentile as example threshold
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold}:")
            display(filtered_df.head())
            norm_field = f"{numeric_field}_normalized"
            filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, norm_field]].head())
            # Try to group by first non-numeric field
            group_fields = [field.id for field in rs.fields if field.id in df.columns and field.data_type not in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number']]
            if group_fields:
                group_field = group_fields[0]
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_' + numeric_field)
                print(f"\nGrouped data by {group_field}:")
                display(grouped_df.head())
        else:
            print(f"Numeric field {numeric_field} not in DataFrame columns.")
    else:
        print("No numeric fields found in the first record set.")
else:
    print("No record sets to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets:
    rs = dataset.record_sets[0]
    df = dataframes[rs.id]
    # Use the same numeric field as in EDA
    numeric_fields = [field.id for field in rs.fields if field.data_type in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number']]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        if numeric_field in df.columns and not df.empty:
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
            plt.figure(figsize=(8, 4))
            sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
            plt.title(f'Distribution of {numeric_field}')
            plt.xlabel(numeric_field)
            plt.ylabel('Count')
            plt.show()
            # If there is a categorical group field, do a boxplot
            group_fields = [field.id for field in rs.fields if field.id in df.columns and field.data_type not in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number']]
            if group_fields:
                group_field = group_fields[0]
                plt.figure(figsize=(10, 4))
                sns.boxplot(x=group_field, y=numeric_field, data=df)
                plt.title(f'{numeric_field} by {group_field}')
                plt.xticks(rotation=45)
                plt.show()
    else:
        print('No numeric fields available for visualization.')
else:
    print('No record sets available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to explore a Croissant-described dataset. We loaded metadata, inspected available record sets and fields by their `@id`, loaded records into dataframes, performed basic filtering and normalization of numeric fields, grouped data by categories, and visualized distributions. Further analyses can build on these foundations using the explicit field identifiers for reproducibility and clarity.